# Descriptive statistics and correlations

This notebook combines all explanatory variables into one table per municipality, summarises them and shows how they correlate with charging point density and with each other. The correlation table is the starting point for the regression in notebook 7: strongly correlated explanatory variables point to possible multicollinearity.

**Folder structure:** this notebook sits in `notebooks/` and reads from `data/raw/`. Outputs go to `data/processed/` and `figures/`.

**Input files in `data/raw/`:**
- `laadpunten_per_gemeente.csv`: regular public charging points per municipality, July 2026 (NAL)
- `inkomens_per_gemeente.csv`: households and standardised income per municipality, 2024 (CBS 86161NED)
- `jongeren_per_gemeente_2026.csv` and `ouderen_per_gemeente_2026.csv`: age structure (CBS)
- `tk2025_uitslag.csv`: results of the Tweede Kamer election of 29 October 2025 (Kiesraad)
- `kwb2024_per_gemeente.csv`: education, population density and share of apartments per municipality, extracted from CBS Kerncijfers wijken en buurten 2024 (85984NED)

**Outputs:**
- `data/processed/all_variables_per_municipality.csv`: all variables in one table, input for notebooks 6.2, 7 and 8
- `data/processed/table_comparison_factors.csv`: correlation of each factor with charging point density
- `figures/figure_correlationmatrix.png` and `figures/figure_distribution_of_public_charging_point_density.png`

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

RAW = "../data/raw/"
PROCESSED = "../data/processed/"
FIGURES = "../figures/"

## 1. Load and merge the data

The data comes from three sources (NAL, CBS and the Kiesraad), which write some municipality names differently, for example *'s-Gravenhage* instead of *Den Haag*. We use the names of the charging point file everywhere. The function `clean_names` removes spaces around the names and replaces the different spellings.

In [ ]:
NAME_MAPPING = {
    "'s-Gravenhage (gemeente)": "Den Haag",
    "'s-Gravenhage": "Den Haag",
    "Groningen (gemeente)": "Groningen",
    "Utrecht (gemeente)": "Utrecht",
    "Beek (L.)": "Beek",
    "Stein (L.)": "Stein",
    "Hengelo (O.)": "Hengelo",
    "Hengelo (O)": "Hengelo",
    "Laren (NH.)": "Laren",
    "Middelburg (Z.)": "Middelburg",
    "Rijswijk (ZH.)": "Rijswijk",
    "Bergen (L)": "Bergen (L.)",
    "Bergen (NH)": "Bergen (NH.)",
}


def clean_names(names):
    """Remove spaces around the municipality names and use the spelling of the charging point file."""
    return names.str.strip().replace(NAME_MAPPING)

### 1a. Charging points, households and income

Charging point density = number of charging points / number of households × 1,000. The CBS file gives households in thousands, so dividing the charging points by this number directly gives charging points per 1,000 households.

In [ ]:
# Charging points (NAL, July 2026)
charging = pd.read_csv(RAW + "laadpunten_per_gemeente.csv", sep=";")
charging.columns = ["gemeente", "laadpunten"]
charging = charging[charging["gemeente"] != "Totaal"]       # remove the national total
charging["gemeente"] = clean_names(charging["gemeente"])

# Households and income (CBS 86161NED, 2024)
income = pd.read_csv(RAW + "inkomens_per_gemeente.csv", sep=";", encoding="utf-8-sig")
for col in income.columns[4:]:                              # numbers are stored as text
    income[col] = pd.to_numeric(income[col].astype(str).str.strip(), errors="coerce")

income = income.rename(columns={
    "Regio's": "gemeente",
    "Particuliere huishoudens (x 1 000)": "huishoudens_x1000",
    "Inkomen/Gemiddeld gestandaardiseerd inkomen (1 000 euro)": "gem_gestandaardiseerd_inkomen_keur",
    "Inkomen/Mediaan gestandaardiseerd inkomen (1 000 euro)": "med_gestandaardiseerd_inkomen_keur",
})
income["gemeente"] = clean_names(income["gemeente"])

# Merge and calculate charging point density
df = charging.merge(income, on="gemeente", how="inner")
df["laadpunten_per_1000_huishoudens"] = df["laadpunten"] / df["huishoudens_x1000"]
df = df[["gemeente", "laadpunten_per_1000_huishoudens",
         "gem_gestandaardiseerd_inkomen_keur", "med_gestandaardiseerd_inkomen_keur"]]

print("Municipalities:", len(df))

### 1b. Age structure

The CBS files already contain percentages of the total population; we keep only the two total columns.

In [ ]:
young = pd.read_csv(RAW + "jongeren_per_gemeente_2026.csv", sep=";", decimal=",", encoding="utf-8-sig")
young = young[["Gemeente", "Totaal, 0 tot 25"]]
young.columns = ["gemeente", "jongeren_onder_25_pct"]
young["gemeente"] = clean_names(young["gemeente"])

old = pd.read_csv(RAW + "ouderen_per_gemeente_2026.csv", sep=";", decimal=",", encoding="utf-8-sig")
old = old[["Gemeente", "Totaal, 65 jaar of ouder"]]
old.columns = ["gemeente", "ouderen_65plus_pct"]
old["gemeente"] = clean_names(old["gemeente"])

df = df.merge(young, on="gemeente", how="left")
df = df.merge(old, on="gemeente", how="left")

### 1c. Left-wing vote share

From the Kiesraad results we keep the number of votes per party list per municipality (`RegioCode` starts with `G`). The left-wing vote share is the number of votes for left-wing parties divided by all votes for party lists × 100.

- **Broad (main definition):** GroenLinks-PvdA, SP, Partij voor de Dieren, DENK, BIJ1, D66 and Volt
- **Narrow (robustness check):** the same without D66 and Volt

In [ ]:
LEFT_NARROW = ["GROENLINKS / Partij van de Arbeid (PvdA)", "SP (Socialistische Partij)",
               "Partij voor de Dieren", "DENK", "BIJ1"]
LEFT_BROAD = LEFT_NARROW + ["D66", "Volt"]

tk = pd.read_csv(RAW + "tk2025_uitslag.csv", sep=";", dtype=str)
tk = tk[tk["RegioCode"].str.startswith("G") & (tk["VeldType"] == "LijstAantalStemmen")].copy()
tk["Waarde"] = tk["Waarde"].astype(int)

# One row per municipality, one column per party
per_party = tk.pivot_table(index="Regio", columns="LijstNaam", values="Waarde", aggfunc="sum").fillna(0)
total_votes = per_party.sum(axis=1)

votes = pd.DataFrame()
votes["gemeente"] = clean_names(pd.Series(per_party.index))
votes["linkse_stemmen_pct"] = (per_party[LEFT_BROAD].sum(axis=1) / total_votes * 100).values
votes["linkse_stemmen_smal_pct"] = (per_party[LEFT_NARROW].sum(axis=1) / total_votes * 100).values

df = df.merge(votes, on="gemeente", how="left")

### 1d. Education, population density and share of apartments

These come from CBS Kerncijfers wijken en buurten 2024. The full CBS download is large (about 72 MB), so a small extract with one row per municipality is kept in `data/raw/kwb2024_per_gemeente.csv`. The next cell only rebuilds that extract if it is missing and the full download is placed in `data/raw/kwb2024/`; normally it does nothing.

In [ ]:
KWB_MEASURES = {
    "M000100": "bevolkingsdichtheid",   # inhabitants per km² land
    "ZW10340": "meergezinswoning_pct",  # apartments as % of housing stock
    "2018700": "opl_laag",              # persons 15-75, low education (basisonderwijs, vmbo, mbo1)
    "2018740": "opl_middel",            # persons 15-75, middle education (havo, vwo, mbo2-4)
    "2018790": "opl_hoog",              # persons 15-75, high education (hbo, wo)
}
EXTRACT = RAW + "kwb2024_per_gemeente.csv"

if not os.path.exists(EXTRACT) and os.path.exists(RAW + "kwb2024/Observations.csv"):
    obs = pd.read_csv(RAW + "kwb2024/Observations.csv", sep=";", dtype=str)
    obs = obs[obs["WijkenEnBuurten"].str.startswith("GM") & obs["Measure"].isin(KWB_MEASURES)]
    extract = obs.pivot(index="WijkenEnBuurten", columns="Measure", values="Value").rename(columns=KWB_MEASURES)
    extract = extract.apply(pd.to_numeric, errors="coerce")
    codes = pd.read_csv(RAW + "kwb2024/WijkenEnBuurtenCodes.csv", sep=";", dtype=str).set_index("Identifier")
    extract.insert(0, "gemeente", codes.loc[extract.index, "Title"].str.strip().values)
    extract.index.name = "gemeentecode"
    extract.reset_index().to_csv(EXTRACT, index=False)
    print("Saved:", EXTRACT)

Highly educated = persons aged 15–75 with hbo or wo, divided by all persons aged 15–75 with a known education level × 100.

In [ ]:
kwb = pd.read_csv(EXTRACT)
kwb["gemeente"] = clean_names(kwb["gemeente"])

educated = kwb["opl_laag"].fillna(0) + kwb["opl_middel"].fillna(0) + kwb["opl_hoog"].fillna(0)
kwb["hoogopgeleid_pct"] = kwb["opl_hoog"] / educated * 100

df = df.merge(kwb[["gemeente", "hoogopgeleid_pct", "bevolkingsdichtheid", "meergezinswoning_pct"]],
              on="gemeente", how="left")

print("Municipalities:", len(df))
print("Variables:", list(df.columns[1:]))

### 1e. Readable labels and missing values

For tables and figures the columns get readable English labels. Then we count the missing values per variable: these are small municipalities for which CBS does not publish income figures.

In [ ]:
LABELS = {
    "laadpunten_per_1000_huishoudens": "Charging points per 1,000 households",
    "gem_gestandaardiseerd_inkomen_keur": "Mean standardised income (k€)",
    "med_gestandaardiseerd_inkomen_keur": "Median standardised income (k€)",
    "jongeren_onder_25_pct": "Residents under 25 (%)",
    "ouderen_65plus_pct": "Residents 65+ (%)",
    "linkse_stemmen_pct": "Left-wing vote share (%)",
    "linkse_stemmen_smal_pct": "Left-wing vote share, narrow (%)",
    "hoogopgeleid_pct": "Highly educated (%)",
    "bevolkingsdichtheid": "Population density (inh./km²)",
    "meergezinswoning_pct": "Apartments (% of dwellings)",
}
data = df[list(LABELS)].rename(columns=LABELS)

missing = data.isna().sum()
print(missing[missing > 0])

## 2. Descriptive statistics

The table shows the number of municipalities with data (N), the mean, the spread and the range of each variable.

In [ ]:
summary = data.describe().T
summary = summary.rename(columns={"count": "N", "std": "SD", "50%": "median"})
summary["N"] = summary["N"].astype(int)
summary.round(2)

In [ ]:
# Distribution of the dependent variable
y = data["Charging points per 1,000 households"]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y, bins=30, color="#1d639b", edgecolor="white")
ax.axvline(y.median(), color="#d32f2f", linestyle="--", label=f"Median = {y.median():.1f}")
ax.set_xlabel("Charging points per 1,000 households")
ax.set_ylabel("Number of municipalities")
ax.set_title(f"Distribution of public charging point density (N = {y.notna().sum()})")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES + "figure_distribution_of_public_charging_point_density.png", dpi=200)
plt.show()

## 3. Comparison of all factors

For each explanatory variable we calculate:
- the **Pearson correlation r**: the strength of the linear relation (−1 to +1),
- the **Spearman correlation ρ**: the same based on ranks, so less sensitive to outliers,
- the **p-values**, and **R² = r²**: the share of the variation explained by this factor alone.

Municipalities with a missing value are left out per pair, so N differs per variable.

In [ ]:
GROUPS = {
    "Mean standardised income (k€)": "Socioeconomic",
    "Median standardised income (k€)": "Socioeconomic",
    "Highly educated (%)": "Socioeconomic",
    "Residents under 25 (%)": "Demographic",
    "Residents 65+ (%)": "Demographic",
    "Population density (inh./km²)": "Need",
    "Apartments (% of dwellings)": "Need",
    "Left-wing vote share (%)": "Political",
    "Left-wing vote share, narrow (%)": "Political",
}
target = "Charging points per 1,000 households"

rows = []
for col in data.columns:
    if col == target:
        continue
    pair = data[[col, target]].dropna()
    r, p = stats.pearsonr(pair[col], pair[target])
    rho, p_rho = stats.spearmanr(pair[col], pair[target])
    rows.append({"Variable": col, "Group": GROUPS[col], "N": len(pair),
                 "Pearson r": r, "p (Pearson)": p,
                 "Spearman ρ": rho, "p (Spearman)": p_rho,
                 "R²": r ** 2})

corr_target = pd.DataFrame(rows)

# Sort from strongest to weakest relation (largest absolute r first)
corr_target["abs_r"] = corr_target["Pearson r"].abs()
corr_target = corr_target.sort_values("abs_r", ascending=False).drop(columns="abs_r")
corr_target.round(3)

In [ ]:
# Save the table for the report
corr_target.round({"Pearson r": 3, "Spearman ρ": 3, "R²": 3}).to_csv(PROCESSED + "table_comparison_factors.csv", index=False)
print("Saved:", PROCESSED + "table_comparison_factors.csv")

## 4. Correlation matrix of all variables

The matrix shows how all variables relate to each other. A diverging colour map is used, because correlations run from −1 (blue) through 0 (white) to +1 (red). Correlations above about 0.7 between two explanatory variables mean they measure partly the same thing, which can cause multicollinearity in the regression.

In [ ]:
corr_matrix = data.corr(method="pearson")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={"label": "Pearson r"}, ax=ax)
ax.set_title("Correlation matrix of all variables")
plt.xticks(rotation=40, ha="right")
plt.tight_layout()
plt.savefig(FIGURES + "figure_correlationmatrix.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Pairs of explanatory variables with |r| > 0.7 (possible multicollinearity)
explanatory = [col for col in data.columns if col != target]

pairs = []
for i in range(len(explanatory)):
    for j in range(i + 1, len(explanatory)):          # each pair only once
        a, b = explanatory[i], explanatory[j]
        r = corr_matrix.loc[a, b]
        if abs(r) > 0.7:
            pairs.append({"Variable 1": a, "Variable 2": b, "Pearson r": round(r, 2)})

pd.DataFrame(pairs)

## 5. Save the merged table

The merged table is the input for notebooks 6.2, 7 and 8.

In [ ]:
df.to_csv(PROCESSED + "all_variables_per_municipality.csv", index=False)
print("Saved:", PROCESSED + "all_variables_per_municipality.csv")